In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# ====================== LOAD DATA ======================
df = pd.read_csv('Churn_Modelling.csv')

print("Dataset Shape:", df.shape)
print("Churn Rate: {:.2f}%".format(df['Exited'].mean() * 100))

# ====================== PREPROCESSING ======================
X = df.drop(['RowNumber', 'CustomerId', 'Surname', 'Exited'], axis=1)
y = df['Exited']

le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])
X = pd.get_dummies(X, columns=['Geography'], drop_first=True)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ====================== MODEL TRAINING ======================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

print("\n" + "="*60)
print("MODEL PERFORMANCE (Random Forest)")
print("="*60)
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))


import joblib
joblib.dump(model, 'churn_random_forest.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')   # Save label encoder too
print("\n✅ Model and preprocessors saved successfully!")


def predict_customer_churn(customer_dict):
    df_new = pd.DataFrame([customer_dict])
    
    # Preprocessing
    df_new['Gender'] = le.transform(df_new['Gender'])
    df_new = pd.get_dummies(df_new, columns=['Geography'], drop_first=True)
    
    # Align columns
    for col in X.columns:
        if col not in df_new.columns:
            df_new[col] = 0
    df_new = df_new[X.columns]
    
    # Scale and Predict
    scaled = scaler.transform(df_new)
    prob = model.predict_proba(scaled)[0][1]
    
    prediction = "Likely to **CHURN**" if prob > 0.5 else "Likely to **STAY**"
    return prediction, round(prob * 100, 2)


print("\n✅ Code is ready to use!")

Dataset Shape: (10000, 14)
Churn Rate: 20.37%

MODEL PERFORMANCE (Random Forest)
Accuracy : 0.8505

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.91      0.91      1593
           1       0.64      0.60      0.62       407

    accuracy                           0.85      2000
   macro avg       0.77      0.76      0.76      2000
weighted avg       0.85      0.85      0.85      2000


Confusion Matrix:
[[1456  137]
 [ 162  245]]

Top 10 Important Features:
             Feature  Importance
2                Age    0.299105
5      NumOfProducts    0.172083
4            Balance    0.129979
8    EstimatedSalary    0.102264
0        CreditScore    0.099365
3             Tenure    0.058262
7     IsActiveMember    0.047138
9  Geography_Germany    0.044079
1             Gender    0.021566
6          HasCrCard    0.013342

✅ Model and preprocessors saved successfully!

✅ Code is ready to use!
